In [23]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tvDatafeed import TvDatafeed, Interval
from ta.momentum import RSIIndicator



In [24]:


# Login no TradingView
tv = TvDatafeed()

ticker = 'WDO1!'
exchange = 'BMFBOVESPA'


df = tv.get_hist(
    symbol=ticker,
    exchange=exchange,
    interval=Interval.in_4_hour,
    n_bars=10000
)
df = df[df.index.year >=2000].dropna()
# df.index = pd.to_datetime(df.index).normalize().date
df.drop(columns='symbol',inplace=True)
# df.index = pd.to_datetime(df.index).normalize()
df.dropna(inplace=True)
df['ret'] = df['close'].pct_change()
df.tail()




,open,high,low,close,volume,ret
datetime,,,,,,
2026-01-02 13:00:00,5475.0,5480.0,5454.0,5460.5,600657.0,-0.002648
2026-01-02 17:00:00,5461.0,5466.0,5452.0,5456.0,116066.0,-0.000824
2026-01-05 09:00:00,5455.0,5489.5,5449.0,5450.5,1386987.0,-0.001008
2026-01-05 13:00:00,5450.5,5455.5,5431.5,5441.5,654548.0,-0.001651
2026-01-05 17:00:00,5441.5,5449.0,5440.0,5449.0,140013.0,0.001378


In [28]:
rsi_ind = RSIIndicator(
    close=df["close"],
    window=14
)

df["rsi"] = rsi_ind.rsi()

# =========================
# CRUZAMENTOS DE ZONA
# =========================
df["buy_signal"] = (df["rsi"] < 30) & (df["rsi"].shift(1) >= 30)
df["sell_signal"] = (df["rsi"] > 70) & (df["rsi"].shift(1) <= 70)

df["back_inside_range"] = (
    (df["rsi"] >= 30) & (df["rsi"] <= 70)
)

# =========================
# POSIÇÃO (regime)
# =========================
position = 0
positions = []

for cross_low, cross_high, inside in zip(
    df["buy_signal"],
    df["sell_signal"],
    df["back_inside_range"]
):
    if position == 0:
        if cross_low:
            position = 1      # entra comprado
        elif cross_high:
            position = -1     # entra vendido

    elif position == 1:
        if inside:
            position = 0      # sai do long ao voltar ao range

    elif position == -1:
        if inside:
            position = 0      # sai do short ao voltar ao range

    positions.append(position)

df["position"] = positions


df["strategy_ret"] = df["position"].shift(1) * df["ret"]


df["strategy"] = df["strategy_ret"].cumsum()
df["buy_hold"] = df["ret"].cumsum()
df.tail()

,open,high,low,close,volume,ret,rsi,cross_below_30,cross_above_70,back_inside_range,position,strategy_ret,strategy,buy_hold,buy_signal,sell_signal
datetime,,,,,,,,,,,,,,,,
2026-01-02 13:00:00,5475.0,5480.0,5454.0,5460.5,600657.0,-0.002648,35.215322,False,False,True,0,-0.0,0.357671,0.431780,False,False
2026-01-02 17:00:00,5461.0,5466.0,5452.0,5456.0,116066.0,-0.000824,34.455026,False,False,True,0,-0.0,0.357671,0.430956,False,False
2026-01-05 09:00:00,5455.0,5489.5,5449.0,5450.5,1386987.0,-0.001008,33.502956,False,False,True,0,-0.0,0.357671,0.429948,False,False
2026-01-05 13:00:00,5450.5,5455.5,5431.5,5441.5,654548.0,-0.001651,31.947295,False,False,True,0,-0.0,0.357671,0.428296,False,False
2026-01-05 17:00:00,5441.5,5449.0,5440.0,5449.0,140013.0,0.001378,34.669683,False,False,True,0,0.0,0.357671,0.429675,False,False


In [29]:
df.tail(30)

,open,high,low,close,volume,ret,rsi,cross_below_30,cross_above_70,back_inside_range,position,strategy_ret,strategy,buy_hold,buy_signal,sell_signal
datetime,,,,,,,,,,,,,,,,
2025-12-17 09:00:00,5516.0,5539.5,5498.5,5536.0,1911761.0,0.001357,69.403326,False,False,True,0,0.000000,0.354658,0.445378,False,False
2025-12-17 13:00:00,5536.0,5545.0,5521.0,5535.5,862130.0,-0.000090,69.249997,False,False,True,0,-0.000000,0.354658,0.445287,False,False
2025-12-17 17:00:00,5536.0,5539.5,5522.5,5534.5,160353.0,-0.000181,68.922040,False,False,True,0,-0.000000,0.354658,0.445107,False,False
2025-12-18 09:00:00,5519.0,5571.5,5514.5,5525.0,1996819.0,-0.001717,65.736998,False,False,True,0,-0.000000,0.354658,0.443390,False,False
2025-12-18 13:00:00,5525.0,5543.5,5513.5,5535.0,813023.0,0.001810,67.442564,False,False,True,0,0.000000,0.354658,0.445200,False,False
2025-12-18 17:00:00,5535.0,5537.5,5529.5,5534.5,126010.0,-0.000090,67.262275,False,False,True,0,-0.000000,0.354658,0.445110,False,False
2025-12-19 09:00:00,5535.0,5556.5,5508.0,5522.0,1642109.0,-0.002259,62.746335,False,False,True,0,-0.000000,0.354658,0.442851,False,False
2025-12-19 13:00:00,5522.0,5543.5,5513.0,5540.0,605603.0,0.003260,66.259329,False,False,True,0,0.000000,0.354658,0.446111,False,False
2025-12-19 17:00:00,5540.0,5560.5,5537.5,5555.0,180444.0,0.002708,68.891930,False,False,True,0,0.000000,0.354658,0.448818,False,False


In [30]:
df_plot = df.tail(4000)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=(f"{ticker} Close Price", "RSI (14)")
)

# =========================
# CLOSE PRICE
# =========================
fig.add_trace(
    go.Scatter(
        x=df_plot.index,
        y=df_plot["close"],
        name="Close Price",
        line=dict(color="black")
    ),
    row=1, col=1
)

# Buy markers (RSI < 30)
fig.add_trace(
    go.Scatter(
        x=df_plot.index[df_plot["buy_signal"]],
        y=df_plot.loc[df_plot["buy_signal"], "close"],
        mode="markers",
        name="Buy (RSI < 30)",
        marker=dict(symbol="triangle-up", color="green", size=10)
    ),
    row=1, col=1
)

# Sell markers (RSI > 70)
fig.add_trace(
    go.Scatter(
        x=df_plot.index[df_plot["sell_signal"]],
        y=df_plot.loc[df_plot["sell_signal"], "close"],
        mode="markers",
        name="Sell (RSI > 70)",
        marker=dict(symbol="triangle-down", color="red", size=10)
    ),
    row=1, col=1
)

# =========================
# RSI
# =========================
fig.add_trace(
    go.Scatter(
        x=df_plot.index,
        y=df_plot["rsi"],
        name="RSI",
        line=dict(color="blue")
    ),
    row=2, col=1
)

# RSI reference lines
fig.add_hline(y=70, line=dict(color="red", dash="dash"), row=2, col=1)
fig.add_hline(y=30, line=dict(color="green", dash="dash"), row=2, col=1)

# Buy / Sell markers on RSI
fig.add_trace(
    go.Scatter(
        x=df_plot.index[df_plot["buy_signal"]],
        y=df_plot.loc[df_plot["buy_signal"], "rsi"],
        mode="markers",
        name="Buy Signal",
        marker=dict(symbol="triangle-up", color="green", size=10)
    ),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(
        x=df_plot.index[df_plot["sell_signal"]],
        y=df_plot.loc[df_plot["sell_signal"], "rsi"],
        mode="markers",
        name="Sell Signal",
        marker=dict(symbol="triangle-down", color="red", size=10)
    ),
    row=2, col=1
)

# =========================
# LAYOUT
# =========================
fig.update_layout(
    title_text=f"{ticker} | Close Price and RSI Strategy",
    height=700,
    template="plotly_white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text="RSI", range=[0, 100], row=2, col=1)

fig.show()


In [31]:
# =========================
# PLOT (styled to match previous Plotly figure)
# =========================
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.65, 0.25],
    subplot_titles=(f"{ticker} | Cumulative Returns", "Position")
)

# Buy & Hold
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["buy_hold"] * 100,
        name="Buy & Hold",
        line=dict(width=2, color="black")
    ),
    row=1, col=1
)

# Strategy
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["strategy"] * 100,
        name="Strategy",
        line=dict(width=2, color="blue")
    ),
    row=1, col=1
)

# Position (filled area)
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["position"],
        name="Position",
        mode="lines",
        line=dict(width=1, color="green"),
        fill="tozeroy",
        fillcolor="rgba(0,200,0,0.08)"
    ),
    row=2, col=1
)

# Layout
fig.update_layout(
    title_text=f"{ticker} | MACD",
    template="plotly_white",
    height=700,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.update_yaxes(title_text="Cumulative Return (%)", row=1, col=1)
fig.update_yaxes(title_text="Position", row=2, col=1, range=[-0.05, 1.05])
fig.update_xaxes(showticklabels=False, row=1, col=1)

fig.show()




In [32]:
total_return_bh = df["buy_hold"].iloc[-1]
total_return_strategy = df["strategy"].iloc[-1]

vol_strategy = df["strategy_ret"].std() * np.sqrt(252)
vol_bh = df["ret"].std() * np.sqrt(252)

sharpe_strategy = (
    df["strategy_ret"].mean() / df["strategy_ret"].std()
) * np.sqrt(252)

sharpe_bh = (
    df["ret"].mean() / df["ret"].std()
) * np.sqrt(252)

print("=== RESULTADOS ===")
print(f"Buy & Hold Return: {total_return_bh:.2%}")
print(f"Strategy Return:   {total_return_strategy:.2%}")
print()
print(f"Buy & Hold Vol: {vol_bh:.2%}")
print(f"Strategy Vol:   {vol_strategy:.2%}")
print()
print(f"Buy & Hold Sharpe: {sharpe_bh:.2f}")
print(f"Strategy Sharpe:   {sharpe_strategy:.2f}")


=== RESULTADOS ===
Buy & Hold Return: 42.97%
Strategy Return:   35.77%

Buy & Hold Vol: 8.90%
Strategy Vol:   3.56%

Buy & Hold Sharpe: 0.23
Strategy Sharpe:   0.48
